Cell 1

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import ParameterSampler

Cell 2

In [2]:
BASE_DIR = Path(
    r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting"
)

ADVANCED_DIR = BASE_DIR / "data" / "advanced"

train = pd.read_csv(
    ADVANCED_DIR / "train_advanced.csv"
)

train["date"] = pd.to_datetime(
    train["date"]
)

print(train.shape)

(3000888, 29)


Cell 3

In [3]:
target = "sales"

features = [
    "store_nbr",
    "family",
    "onpromotion",

    "year",
    "month",
    "day",
    "quarter",
    "week",
    "day_of_week",
    "is_weekend",

    "city",
    "state",
    "type",
    "cluster",

    "dcoilwtico",
    "is_payday",
    "is_national_holiday",
    "earthquake_period",

    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",

    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",

    "rolling_std_7",
    "rolling_std_28"
]

X = train[features]

y = np.log1p(train[target])

X = pd.get_dummies(
    X,
    drop_first=True
)

Cell 4

In [4]:
split_date = "2017-01-01"

train_mask = train["date"] < split_date
valid_mask = train["date"] >= split_date

X_train = X.loc[train_mask]
X_valid = X.loc[valid_mask]

y_train = y.loc[train_mask]

y_valid_original = train.loc[
    valid_mask,
    "sales"
]

Cell 5

In [5]:
sample_idx = X_train.sample(
    n=500000,
    random_state=42
).index

X_train_small = X_train.loc[sample_idx]
y_train_small = y_train.loc[sample_idx]

print(X_train_small.shape)

(500000, 94)


Cell 6

In [6]:
param_grid = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "n_estimators": [300, 500, 700]
}

Cell 7

In [7]:
param_list = list(
    ParameterSampler(
        param_grid,
        n_iter=10,
        random_state=42
    )
)

Cell 8

In [8]:
results = []

for params in param_list:

    print("Testing:", params)

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        **params
    )

    model.fit(
        X_train_small,
        y_train_small
    )

    preds_log = model.predict(
        X_valid
    )

    preds = np.expm1(
        preds_log
    )

    preds = np.maximum(
        preds,
        0
    )

    rmsle = np.sqrt(
        mean_squared_log_error(
            y_valid_original,
            preds
        )
    )

    results.append({
        **params,
        "rmsle": rmsle
    })

    print("RMSLE:", rmsle)

Testing: {'subsample': 0.7, 'n_estimators': 700, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
RMSLE: 0.4621104225731383
Testing: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
RMSLE: 0.49151216326197494
Testing: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
RMSLE: 0.46978953563915393
Testing: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
RMSLE: 0.4725188312276487
Testing: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
RMSLE: 0.47658175495767896
Testing: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
RMSLE: 0.4660484517827139
Testing: {'subsample': 0.7, 'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
RMSLE: 0.48883798797867195
Testing: {'subsample': 

Cell 9

In [9]:
results_df = pd.DataFrame(
    results
)

results_df.sort_values(
    "rmsle"
).head(10)

,subsample,n_estimators,max_depth,learning_rate,colsample_bytree,rmsle
8,1.0,500,6,0.05,1.0,0.456175
7,0.8,700,6,0.10,0.8,0.458607
0,0.7,700,8,0.03,0.8,0.462110
5,0.7,300,8,0.03,0.8,0.466048
2,1.0,300,10,0.03,0.8,0.469790
3,0.7,300,6,0.03,0.7,0.472519
4,0.7,300,4,0.10,0.8,0.476582
9,0.7,300,4,0.05,0.8,0.485386
6,0.7,500,4,0.03,1.0,0.488838
1,0.7,300,4,0.03,0.8,0.491512
